In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from wordcloud import WordCloud

# Thiết lập style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'sans-serif']

# 1. Tải dữ liệu

In [ ]:
# Đọc tất cả các tập dữ liệu
train_df = pd.read_csv('../data/processed_3labels/train.csv', encoding='utf-8')
val_df = pd.read_csv('../data/processed_3labels/val.csv', encoding='utf-8')
test_df = pd.read_csv('../data/processed_3labels/test.csv', encoding='utf-8')

# Gộp dữ liệu để phân tích chung
df = pd.concat([train_df, val_df, test_df], ignore_index=True)

print(f"Tổng số mẫu dữ liệu: {len(df)}")
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nMột số dòng dữ liệu đầu tiên:")
df.head()

# 2. Phân bố nhãnn

In [ ]:
# Đếm số lượng nhãn
label_counts = df['emotion'].value_counts()
print("Phân bố nhãn:")
print(label_counts)
print(f"\nPhân trăm:")
print(label_counts / len(df) * 100)

In [ ]:
# Vẽ biểu đồ cột
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Biểu đồ cột
colors = {'POSITIVE': '#4CAF50', 'NEUTRAL': '#9E9E9E', 'NEGATIVE': '#F44336'}
label_counts.plot(kind='bar', ax=ax1, color=[colors[l] for l in label_counts.index])
ax1.set_title('Phân phối nhãn (Count)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Sentiment', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.tick_params(axis='x', rotation=0)

# Biểu đồ tròn
label_counts.plot(kind='pie', ax=ax2, autopct='%1.1f%%', 
                  colors=[colors[l] for l in label_counts.index],
                  startangle=90)
ax2.set_title('Phân phối nhãn (%)', fontsize=14, fontweight='bold')
ax2.set_ylabel('')

plt.tight_layout()
plt.show()

# 3. Phân tích dữ liệu

In [ ]:
# Tính độ dài văn bản
df['text_length'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

print("Thống kê độ dài văn bản:")
print(df[['text_length', 'word_count']].describe())

In [ ]:
# Vẽ biểu đồ độ dài văn bản cho từng loại cảm xúc
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Độ dài văn bản
for label in ['POSITIVE', 'NEUTRAL', 'NEGATIVE']:
    data = df[df['emotion'] == label]['text_length']
    ax1.hist(data, alpha=0.6, label=label, bins=30, color=colors[label])
ax1.set_title('Phân bố độ dài văn bản (Characters)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Characters', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.legend()

# Đếm từ
for label in ['POSITIVE', 'NEUTRAL', 'NEGATIVE']:
    data = df[df['emotion'] == label]['word_count']
    ax2.hist(data, alpha=0.6, label=label, bins=20, color=colors[label])
ax2.set_title('Word Count Distribution', fontsize=14, fontweight='bold')
ax2.set_xlabel('Words', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.legend()

plt.tight_layout()
plt.show()

# 4. Ví dụ

In [ ]:
# Hiển thị 3 ví dụ cho mỗi nhãn
print("Mẫu dữ liệu:\n")
for label in ['POSITIVE', 'NEUTRAL', 'NEGATIVE']:
    print(f"\n{'='*60}")
    print(f"{label}:")
    print('='*60)
    samples = df[df['emotion'] == label]['text'].sample(3, random_state=42)
    for i, text in enumerate(samples, 1):
        print(f"{i}. {text}")

# 5. Kiểm tra dữ liệu

In [ ]:
# Kiểm tra giá trị thiếu
print("Missing values:")
print(df.isnull().sum())

# Kiểm tra giá trị trùng
duplicates = df.duplicated(subset=['text']).sum()
print(f"\nDuplicate texts: {duplicates}")

# Kiểm tra văn bản quá ngắn
very_short = df[df['word_count'] < 3]
print(f"\nTexts with < 3 words: {len(very_short)}")
if len(very_short) > 0:
    print("Examples:")
    print(very_short[['text', 'emotion']].head())

# 6. Phân tích dữ liệu

In [ ]:
# kiểm tra dữ liệu
print("Phân bố tập train:")
print(train_df['emotion'].value_counts())
print(f"\nPhân bố tập validation:")
print(val_df['emotion'].value_counts())
print(f"\nPhân bố tập test:")
print(test_df['emotion'].value_counts())

In [ ]:
# Vẽ biểu đồ so sánh giữa các tập dữ liệu
split_data = pd.DataFrame({
    'Train': train_df['emotion'].value_counts(),
    'Val': val_df['emotion'].value_counts(),
    'Test': test_df['emotion'].value_counts()
})

split_data.plot(kind='bar', figsize=(10, 6), 
                color=['#2196F3', '#FF9800', '#9C27B0'])
plt.title('Label Distribution Across Splits', fontsize=14, fontweight='bold')
plt.xlabel('Sentiment', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=0)
plt.legend(title='Split')
plt.tight_layout()
plt.show()